# Project Report

# Introduction and description of the dataset


This project tackles a niche but visually interesting problem: distinguishing between
two types of Japanese plush toys based on the *Touhou Project* franchise —
**Fumo** and **Chocopuni**. Both represent the same characters, but differ
significantly in style, shape, and manufacturer.

| | Fumo (FumoFumo) | Chocopuni (Chokopuni) |
|---|---|---|
| **Manufacturer** | Gift (official) | Good Smile Company |
| **Style** | Classic, slightly angular, "mofumofu" | Round, chibi, soft "puni" shape |
| **Size** | ~20–25 cm (seated) | Smaller, more compact |
| **Origin** | Touhou Project (general) | Touhou LostWord (game) |
| **Face** | Flat face, embroidered eyes | Softer features, printed eyes |

Despite belonging to the same franchise, the two types have distinct silhouettes,
proportions, and textures — making this a realistic fine-grained visual classification task.


## Dataset

The Fumo and Chocopuni datasets consist of approximately 90% pictures taken by myself and friends.
The remaining images were scraped from the internet and used primarily as test data —
the goal being to verify that the model can recognize plushies it has never seen before,
not just the ones it was trained on. This gives a much more honest measure of real-world performance.

📁 **Dataset:** [Link to dataset](https://drive.google.com/drive/folders/1gp_d7FjI8-9Uuu579UwrCWxPdw3DeMX8)




### Dataset Collection

With only 4 chocopunis and 11 fumos available, reaching a meaningful dataset size
without causing overfitting was a real constraint.

Collection happened in three stages:

1. **Controlled environment** — black background, white light, rotating the plushie to capture all angles
2. **Second controlled environment** — white background, warm light, still clean studio-style shots
3. **Random environments** — various rooms, lighting conditions, backgrounds, and positions to simulate real-world photos

This brought the count to roughly ~60 images per class. To go further, I reached out to
friends and acquaintances who also own these plushies, which added more unique subjects
and backgrounds to the dataset.

### Final split

| Subset | Images | Purpose |
|---|---|---|
| Train | ~70% | Model training with augmentation |
| Validation | ~15% | Early stopping and hyperparameter decisions |
| Test | ~15% | Final evaluation only — never seen during training |



## Model Performance Summary

| | Model 1 — Custom CNN | Model 2 — EfficientNet-B0 |
|---|---|---|
| **Test Accuracy** | 61.0% | 85.4% |
| **Loss** | 0.671 | 0.474 |
| **Macro F1** | 0.60 | 0.85 |
| **Chocopuni F1** | 0.65 | 0.85 |
| **Fumo F1** | 0.56 | 0.86 |
| **Bias** | Strong toward Chocopuni | Balanced |

## Model 1 — Custom CNN

The baseline model was a small 3-block convolutional network trained entirely from
scratch. Test accuracy reached **61%** — only slightly above random chance (50%),
confirming the expected outcome for a network with no pretrained knowledge and a
small dataset.

The classification report revealed a clear bias: Chocopuni recall was 0.79 while
Fumo recall was only 0.45 — meaning the model missed more than half of all Fumo
images. The model appeared to latch onto superficial features (possibly background
color or overall brightness) rather than learning the actual shape differences
between the two classes.

The confusion matrix confirmed this: a disproportionate number of Fumo images were
predicted as Chocopuni, suggesting the decision boundary was skewed rather than
learned properly.

---

## Model 2 — EfficientNet-B0 as Feature Extractor

Switching to a pretrained EfficientNet-B0 backbone produced a dramatic improvement.
Test accuracy jumped to **85.4%** — a gain of **+24.4 percentage points** over Model 1.

Key differences from Model 1:
- The frozen EfficientNet layers provided rich, generalizable feature representations
  (edges, textures, shapes) learned from 1.2 million ImageNet images
- Only the last convolutional block and the classifier head were trained on our data,
  drastically reducing the risk of overfitting
- Stratified K-Fold cross-validation (3 folds) replaced the fixed train/val split,
  giving a more reliable estimate of generalization on a small dataset

The class bias present in Model 1 largely disappeared: both classes reached F1
scores of 0.85–0.86, and the gap between Chocopuni recall (0.89) and Fumo recall
(0.82) was small and acceptable.



## Challenges

### Switching to PyTorch

The project started with Keras (PyTorch backend) but encountered kernel crashes
related to the `Permute` layer and multiprocessing conflicts on Windows. From
Model 2 onwards the project moved to pure PyTorch with a manual training loop.
This required learning gradient handling, DataLoader mechanics, and writing
evaluation loops from scratch — but resulted in a more transparent and
debuggable setup.

### Dataset collection

The physical collection was limited by the number of plushies available. Shooting
in multiple environments (controlled studio vs. random real-world) was intentional
but introduced a train/test distribution mismatch: training images were mostly
clean studio shots while test images included real-world backgrounds. This likely
contributed to Model 1's poor generalization.

### Overfitting in Model 1

Training accuracy in Model 1 exceeded 98% within the first few epochs while
validation accuracy stagnated — a textbook case of overfitting on a small dataset.
Dropout (0.5), data augmentation, and early stopping were applied, but without
pretrained features the model still failed to learn meaningful representations.


![Overfitting](images/overfitting.png.png)


The confusion matrix revealed the real problem: the model had a strong bias toward
predicting chocopuni, missing roughly half of all fumo images. This pointed to both
class imbalance and a mismatch between the training and test distributions -
training images were shot on controlled backgrounds, while test images came from
varied real-world settings.







### Windows multiprocessing and memory

Running DataLoader with `NUM_WORKERS > 4` on Windows caused `WinError 1455`
(paging file too small) because each worker spawns a full PyTorch process.
Reducing `NUM_WORKERS` to 4 resolved the issue with no meaningful impact on
training speed given the dataset size.

## Reflection

The results clearly demonstrate that **transfer learning is the correct approach**
for small custom image datasets. Model 1 shows that even a well-regularized CNN
trained from scratch cannot reliably learn the subtle visual differences between
Fumo and Chocopuni with ~300 images. Model 2 achieves strong performance by
reusing features that took millions of images to learn, fine-tuning only a small
fraction of parameters on our specific task.

The remaining 15% error rate in Model 2 is likely due to genuinely ambiguous
images — plushies photographed at unusual angles, with heavy occlusion, or in
low light — rather than a fundamental failure of the model. A larger and more
diverse dataset would be the most direct path to further improvement.